# Bias generator v2: write + repair (Colab)
Runtime → Change runtime type → **T4 GPU**, then **Run all**.

Trains Qwen2.5-1.5B on two jobs (write a bias from scratch; repair a broken one using XHAIL's feedback), then generates biases for three test tiers. Everything is written to Google Drive, so a disconnect loses nothing; scoring happens afterwards on the laptop.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/Research Engineering/refresh-xhail/xhail-colab"
!pip install -q unsloth
!pip install -q -e .
%cd experiments/bias_llm
!wc -l data/sft_train.jsonl data/sft_val.jsonl

## 1. Train (≈60–70 min on a T4)
Skips itself if the adapter already exists, so re-running after a disconnect resumes at evaluation.

In [ ]:
import os
if not os.path.exists('checkpoints/qwen-sft/adapter_config.json'):
    !python train_unsloth.py --sft --model unsloth/Qwen2.5-1.5B-Instruct --epochs 2 --out checkpoints/qwen-sft
else:
    print('adapter found, skipping training')

## 2. Write a bias from scratch, with up to 4 attempts
Tiers: **seen** (160 training programs), **test** (160 unseen synthetic), **gold** (16 hand-written). `resample` retries blind; `feedback` retries with XHAIL's diagnosis. Attempt 1 is greedy and shared, so it doubles as the single-attempt result.

In [ ]:
M = 'checkpoints/qwen-sft'
runs = [('seen', 'data/train.jsonl --limit 160', [0])] + [(n, s, [0, 1, 2]) for n, s in [('test', 'data/test.jsonl'), ('gold', 'gold')]]
for tier, tasks, seeds in runs:
    for strategy in ['resample', 'feedback']:
        for seed in seeds:
            out = f'preds/v2/sft_{tier}_{strategy}_s{seed}.jsonl'
            if not os.path.exists(out):
                !python colab_eval.py --model {M} --tasks {tasks} --strategy {strategy} --seed {seed} --out {out}

## 3. Repair a broken bias in one go
Corrupted biases with XHAIL's feedback, for seen programs, unseen synthetic programs and the gold set.

In [ ]:
for tier, path, limit in [('seen', 'data/repair_train.jsonl', '--limit 300'), ('test', 'data/repair_test.jsonl', '--limit 300'), ('gold', 'data/repair_gold.jsonl', '')]:
    out = f'preds/v2/sft_repair_{tier}.jsonl'
    if not os.path.exists(out):
        !python colab_eval.py --model {M} --repair {path} {limit} --out {out}

## 4. Baseline: the untuned model, 3-shot, same prompt and loop

In [ ]:
for tier, tasks in [('test', 'data/test.jsonl'), ('gold', 'gold')]:
    out = f'preds/v2/base3shot_{tier}_feedback_s0.jsonl'
    if not os.path.exists(out):
        !python colab_eval.py --model unsloth/Qwen2.5-1.5B-Instruct --fewshot 3 --tasks {tasks} --strategy feedback --seed 0 --out {out}
!ls -la preds/v2